In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import pearsonr, spearmanr, kendalltau
from scipy.cluster.hierarchy import linkage, dendrogram

In [ ]:
df = pd.read_csv("Womens Clothing E-Commerce Reviews.csv")

df.head()

In [ ]:
df.shape

In [ ]:
df.columns.tolist()

In [ ]:
df.info()

In [ ]:
numerical_cols = [
    "Age",
    "Rating",
    "Recommended IND",
    "Positive Feedback Count"
]

num_df = df[numerical_cols].copy()

num_df.head()

Calculate Pearson correlation

In [ ]:
pearson_corr = num_df.corr(method="pearson")

print("Pearson Correlation:")
display(pearson_corr)

In [ ]:
plt.figure(figsize=(8, 5))

sns.heatmap(
    pearson_corr,
    annot=True,
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    fmt=".2f"
)

plt.title("Pearson Correlation Heatmap")
plt.show()

Calculate Spearman correlation

In [ ]:
spearman_corr = num_df.corr(method="spearman")

print("Spearman Correlation:")
display(spearman_corr)

In [ ]:
plt.figure(figsize=(8, 5))

sns.heatmap(
    spearman_corr,
    annot=True,
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    fmt=".2f"
)

plt.title("Spearman Correlation Heatmap")
plt.show()

Calculate Kendall correlation

In [ ]:
kendall_corr = num_df.corr(method="kendall")

print("Kendall Correlation:")
display(kendall_corr)

In [ ]:
plt.figure(figsize=(8, 5))

sns.heatmap(
    kendall_corr,
    annot=True,
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    fmt=".2f"
)

plt.title("Kendall Correlation Heatmap")
plt.show()

Dynamically filter significant correlations

In [ ]:
methods = {
    "Pearson": pearsonr,
    "Spearman": spearmanr,
    "Kendall": kendalltau
}

results = []

for method_name, method_func in methods.items():

    for i in range(len(numerical_cols)):
        for j in range(i + 1, len(numerical_cols)):

            col1 = numerical_cols[i]
            col2 = numerical_cols[j]

            temp = num_df[[col1, col2]].dropna()

            corr, p_value = method_func(
                temp[col1],
                temp[col2]
            )

            results.append({
                "Method": method_name,
                "Variable 1": col1,
                "Variable 2": col2,
                "Correlation": corr,
                "P-value": p_value
            })

results_df = pd.DataFrame(results)

results_df["Significant"] = results_df["P-value"] < 0.05

significant_corr = results_df[
    (results_df["P-value"] < 0.05) &
    (results_df["Correlation"].abs() >= 0.10)
].sort_values("P-value")

display(significant_corr.round(4))

Create a hierarchical clustering heatmap

In [ ]:
sns.clustermap(
    spearman_corr,
    annot=True,
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    figsize=(8, 7),
    method="average"
)

plt.suptitle(
    "Hierarchical Clustering of Clothing Dataset",
    y=1.02
)

plt.show()

Compare all three methods

In [ ]:
comparison = results_df.pivot_table(
    index=["Variable 1", "Variable 2"],
    columns="Method",
    values="Correlation"
)

print("Correlation Comparison:")
display(comparison.round(3))

Automatically identify the strongest correlations

In [ ]:
strongest = results_df.loc[
    results_df["Correlation"].abs().idxmax()
]

print("Strongest Correlation:")
print("Method:", strongest["Method"])
print("Variable 1:", strongest["Variable 1"])
print("Variable 2:", strongest["Variable 2"])
print("Correlation:", round(strongest["Correlation"], 3))
print("P-value:", round(strongest["P-value"], 5))

In [ ]:
if not significant_corr.empty:
    print("Strongest Significant Correlation:")
    display(significant_corr.head(1).round(4))
else:
    print("No correlations passed the selected thresholds.")